# Multi-*e*GO — mg reference simulation with GROMACS

This notebook runs the full **molten-globule (mg) reference simulation** required by the multi-eGO
workflow, using GROMACS.  Starting from a PDB file it carries out every setup step:

| Step | Tool | What it does |
|------|------|--------------|
| 3 | `gmx pdb2gmx` | Build the topology from the PDB |
| 4 | `gmx editconf` | Define the periodic simulation box |
| 5 | `mego --egos mg` | Generate the mg force-field parameters |
| 7 | `gmx mdrun` | Energy minimisation |
| 8 | `gmx mdrun` | NVT production run |

The resulting trajectory (`run.xtc`) and run-input file (`run.tpr`) are everything you need for
the next step of the multi-eGO workflow (`cmdata` → `make_mat` → `mego`).

> **Before you start:** go to *Runtime → Change runtime type* and select **GPU** to accelerate
> the production run.

---

## Choose your GROMACS installation (run **one** of the two options below)

| | Option A — apt | Option B — compile |
|-|---------------|--------------------|
| Time | ~2 min | ~10 min |
| Version | Ubuntu-packaged | multi-eGO fork (recommended) |
| GPU support | depends on runtime | yes (CUDA auto-detected) |
| cmdata support | ✗ | ✓ |

Run **either** the *Option A* cell **or** the *Option B* cell — not both.

## ▶ Option A — Install GROMACS via apt  *(fast, ~2 min)*

Installs the GROMACS version packaged by Ubuntu.  Covers all steps of the mg workflow.
Skip this cell and run **Option B** instead if you want the multi-eGO GROMACS fork.

In [ ]:
# ── Option A — run this cell OR the Option B cell below, not both ────────────
!apt update -qq && apt install -y -qq gromacs
!gmx --version | head -5

## ▶ Option B — Compile GROMACS from the multi-eGO fork  *(~10 min, recommended)*

Builds the exact version of GROMACS used during multi-eGO development, with CUDA GPU support
when a GPU runtime is selected.  FFTW3 is installed from apt (faster than compiling it
from source).  **This option is required if you also want to build `cmdata` (step 11).**

Skip this cell if you already ran **Option A** above.

In [ ]:
# ── Option B — run this cell OR the Option A cell above, not both ────────────
import subprocess, os

!apt update -qq && apt install -y -qq cmake build-essential libfftw3-dev ninja-build

# Clone the multi-eGO GROMACS fork (release-2023 branch)
!git clone --depth 1 -b release-2023 https://github.com/multi-ego/gromacs.git gromacs_src 2>/dev/null

# Configure
# - GMX_FFT_LIBRARY=fftw3   : use the system FFTW3 installed above (faster than compiling internally)
# - GMX_INSTALL_LEGACY_API  : required to build cmdata against this GROMACS installation
# - CUDA is picked up automatically if a GPU runtime is active
!cmake -S gromacs_src -B gromacs_src/build -G Ninja \
       -DBUILD_TESTING=OFF \
       -DGMX_FFT_LIBRARY=fftw3 \
       -DGMX_INSTALL_LEGACY_API=ON \
       -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs_mego \
       > /dev/null 2>&1

# Build and install (~10 min)
!ninja -C gromacs_src/build -j$(nproc) 2>/dev/null
!ninja -C gromacs_src/build install 2>/dev/null

# Make gmx available on PATH
subprocess.run(["ln", "-sf", "/usr/local/gromacs_mego/bin/gmx", "/usr/local/bin/gmx"], check=True)
!gmx --version | head -5

## 1 · Install multi-eGO

The only Colab-specific version constraint that needs attention is **pandas**: without an
explicit upper bound, pip resolves to pandas 3.x, which breaks several pre-installed Colab
packages (`google-colab`, `cudf`, `gradio`, …).  The cell below pins pandas to
`>=2.2.3,<3.0.0` before installing multi-eGO so that pip's resolver stays on 2.x.

All other dependencies are installed automatically by `pip install -e multi-ego`.

In [ ]:
import sys, subprocess, os

# ── Pin pandas to <3.0.0 to avoid breaking Colab's pre-installed packages ─────
# Without this, pip resolves to pandas 3.x which is incompatible with
# google-colab, cudf, gradio, bqplot, and db-dtypes.
import pandas as _pd

_cur_pd = tuple(int(x) for x in _pd.__version__.split(".")[:3])

if _cur_pd < (2, 2, 3) or _cur_pd >= (3, 0, 0):
    print(f"Pinning pandas {_pd.__version__} → >=2.2.3,<3.0.0 …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2.3,<3.0.0"])
else:
    print(f"pandas {_pd.__version__} ✓")

# ── Clone and install multi-eGO (all other dependencies installed automatically) ─
!git clone --depth 1 https://github.com/multi-ego/multi-ego.git > /dev/null 2>&1
!pip install -q -e multi-ego

# ── Set GMXLIB so GROMACS and ParmEd find the multi-eGO force-field files ─────
os.environ["GMXLIB"] = os.path.abspath("multi-ego")
print("GMXLIB =", os.environ["GMXLIB"])

## 2 · Choose your input structure

Select one of three ways to provide the starting structure, then run the cell below.

| Tab | When to use |
|-----|-------------|
| 📁 **Upload file** | You have a PDB or GRO file on your computer |
| 🧪 **Test input** | Quick start — use one of the built-in multi-eGO reference structures |
| 🧬 **FASTA sequence** | You only have a sequence — the notebook builds a linear extended chain |

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# ── Tab 0: Upload ──────────────────────────────────────────────────────────────
upload_w = widgets.FileUpload(
    accept=".pdb,.gro", multiple=False, description="Select file", layout=widgets.Layout(width="300px")
)

# ── Tab 1: Test inputs (built-in reference structures from the multi-eGO repo) ─
TEST_INPUTS = {
    "GB1 protein (56 residues)": "multi-ego/tests/test_inputs/gpref/gb1_mego.gro",
    "Aβ42 peptide (42 residues)": "multi-ego/tests/test_inputs/abetaref/ab42_mego.gro",
    "TTR tetramer (94 res / chain)": "multi-ego/tests/test_inputs/ttrref/ttr_mego.gro",
    "Lysozyme (with benzene, 129 res)": "multi-ego/tests/test_inputs/lyso-bnz_ref/lyso_mego.gro",
}
test_w = widgets.Dropdown(
    options=list(TEST_INPUTS.keys()),
    description="Structure:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

# ── Tab 2: FASTA sequence → linear extended chain ─────────────────────────────
fasta_w = widgets.Textarea(
    placeholder="Paste single-letter sequence here (e.g. MQYKLILNGKT…)",
    layout=widgets.Layout(width="500px", height="100px"),
)
sysname_w = widgets.Text(
    value="myprotein",
    description="System name:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)

tab = widgets.Tab(
    children=[
        widgets.VBox([widgets.Label("Upload a PDB or GRO file from your computer:"), upload_w]),
        widgets.VBox([widgets.Label("Choose one of the multi-eGO reference structures:"), test_w]),
        widgets.VBox(
            [
                widgets.Label("Paste a single-letter FASTA sequence (no header line):"),
                fasta_w,
                sysname_w,
            ]
        ),
    ]
)
tab.set_title(0, "📁 Upload file")
tab.set_title(1, "🧪 Test input")
tab.set_title(2, "🧬 FASTA sequence")
display(tab)

In [ ]:
import os, shutil
import numpy as np


# ── FASTA → extended-chain PDB (used only for tab 2) ─────────────────────────
def _fasta_to_pdb(sequence, filename):
    """Build a linear extended-strand backbone PDB from a single-letter sequence.

    Only backbone heavy atoms (N, CA, C, O) are written; gmx pdb2gmx will add
    hydrogen atoms and side-chain atoms from the force-field templates.
    Geometry follows Engh & Huber (1991) ideal values; phi=-120, psi=120 (beta-strand).
    """
    ONE3 = {
        "A": "ALA",
        "C": "CYS",
        "D": "ASP",
        "E": "GLU",
        "F": "PHE",
        "G": "GLY",
        "H": "HIS",
        "I": "ILE",
        "K": "LYS",
        "L": "LEU",
        "M": "MET",
        "N": "ASN",
        "P": "PRO",
        "Q": "GLN",
        "R": "ARG",
        "S": "SER",
        "T": "THR",
        "V": "VAL",
        "W": "TRP",
        "Y": "TYR",
    }

    def nerf(a, b, c, bond, ang, dih):
        """Place atom d: NERF algorithm — dihedral a-b-c-d = dih (degrees)."""
        th, ph = np.radians(ang), np.radians(dih)
        bc = (c - b) / np.linalg.norm(c - b)
        n = np.cross(b - a, bc)
        n /= np.linalg.norm(n)
        return c + bond * (-np.cos(th) * bc + np.sin(th) * np.cos(ph) * np.cross(n, bc) + np.sin(th) * np.sin(ph) * n)

    seq = "".join(c for c in sequence.upper() if c.isalpha())
    bad = set(seq) - set(ONE3)
    if bad:
        raise ValueError(f"Unknown amino acid(s): {', '.join(sorted(bad))}")

    PHI, PSI, OMG = -120.0, 120.0, 180.0

    # Seed first residue
    Ns = [np.zeros(3)]
    CAs = [np.array([1.458, 0.0, 0.0])]
    Cs = [nerf(np.array([-1.0, 0.0, 0.0]), Ns[0], CAs[0], 1.523, 111.0, PHI)]

    # Grow chain residue by residue
    for i in range(len(seq) - 1):
        N1 = nerf(Ns[i], CAs[i], Cs[i], 1.329, 116.6, PSI)
        CA1 = nerf(CAs[i], Cs[i], N1, 1.458, 121.7, OMG)
        C1 = nerf(Cs[i], N1, CA1, 1.523, 111.0, PHI)
        Ns.append(N1)
        CAs.append(CA1)
        Cs.append(C1)

    # Carbonyl oxygens — placed in the backbone plane
    Os = [nerf(Ns[i], CAs[i], Cs[i], 1.231, 120.8, PSI - 180.0) for i in range(len(seq))]
    OXT = nerf(Ns[-1], CAs[-1], Cs[-1], 1.229, 120.8, PSI)

    lines = ["REMARK  Extended-strand backbone built from FASTA by multi-eGO\n"]
    serial = 1
    for i, aa in enumerate(seq):
        resname = ONE3[aa]
        rseq = i + 1
        for aname, pos in [("N", Ns[i]), ("CA", CAs[i]), ("C", Cs[i]), ("O", Os[i])]:
            x, y, z = pos
            lines.append(
                f"ATOM  {serial:5d}  {aname:<4s}{resname} A{rseq:4d}    " f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00\n"
            )
            serial += 1
    x, y, z = OXT
    lines.append(f"ATOM  {serial:5d}  OXT {ONE3[seq[-1]]} A{len(seq):4d}    " f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00\n")
    lines += ["TER\n", "END\n"]
    with open(filename, "w") as f:
        f.writelines(lines)
    return filename


# ── Process the selected tab ──────────────────────────────────────────────────
idx = tab.selected_index

if idx == 0:  # ── Upload ──────────────
    if not upload_w.value:
        raise RuntimeError("No file uploaded — select the '📁 Upload file' tab and upload a file first.")
    info = list(upload_w.value.values())[0]
    filename = info["metadata"]["name"]
    with open(filename, "wb") as f:
        f.write(info["content"])
    SYSTEM = os.path.splitext(os.path.basename(filename))[0]

elif idx == 1:  # ── Test input ──────────
    src = TEST_INPUTS[test_w.value]
    filename = os.path.basename(src)
    shutil.copy(src, filename)
    SYSTEM = os.path.splitext(filename)[0]

else:  # ── FASTA ───────────────
    seq = "".join(c for c in fasta_w.value.upper() if c.isalpha())
    if not seq:
        raise RuntimeError("No sequence entered — select the '🧬 FASTA sequence' tab and paste a sequence.")
    SYSTEM = sysname_w.value.strip() or "myprotein"
    filename = f"{SYSTEM}.pdb"
    _fasta_to_pdb(seq, filename)
    print(f"Built {len(seq)}-residue extended chain → {filename}")

print(f"\nInput file : {filename}")
print(f"System name: {SYSTEM}")

## 3 · Generate GROMACS topology  (`gmx pdb2gmx`)

`pdb2gmx` reads the input file and produces `topol.top` and `processed.gro` using
the **multi-eGO-basic** force field (option 1 in the GMXLIB).  The `-ignh` flag
discards any existing hydrogen atoms so they can be rebuilt consistently from the
force-field templates — this is required when using the pre-processed test-input
GRO files and is harmless otherwise.  No water model is added because the mg
simulation is run without explicit solvent.

In [ ]:
# Select force field 1 (multi-ego-basic) and water model 1 (none)
# -ignh: strip existing H atoms and rebuild from force-field templates
!printf "1\n1\n" | gmx pdb2gmx -f {filename} -o processed.gro -water none -ignh 2>&1 | tail -15

## 4 · Set the simulation box  (`gmx editconf`)

Creates a cubic periodic box centred on the molecule.  The padding (`-d`, in nm)
ensures the protein does not interact with its periodic image.

In [ ]:
BOX_PADDING = 5.0  # nm — increase for larger proteins

!gmx editconf -f processed.gro -o boxed.gro -c -bt cubic -d {BOX_PADDING} 2>&1 | tail -8

## 5 · Generate the mg force field  (`mego --egos mg`)

`mego --egos mg` reads the GROMACS topology and writes the molten-globule (mg)
force-field parameters:

- `topol_mego.top` — GROMACS topology with mg interactions
- `ffnonbonded.itp` — C6/C12 non-bonded parameters (referenced by the topology)

Both files are copied to the working directory.

In [ ]:
import subprocess, shutil, os, glob

# ── Prepare mego input directory ───────────────────────────────────────────────
# mego expects:  <inputs_dir>/<system>/topol.top
inputs_dir = os.path.abspath("mego_inputs")
system_dir = os.path.join(inputs_dir, SYSTEM)
os.makedirs(system_dir, exist_ok=True)
shutil.copy("topol.top", os.path.join(system_dir, "topol.top"))

# Outputs will be written to  <outputs_dir>/<system>/mg_N/
# (mego auto-increments N to avoid overwriting previous runs)
outputs_dir = os.path.abspath("mego_outputs")

# ── Run mego ───────────────────────────────────────────────────────────────────
result = subprocess.run(
    [
        "mego",
        "--system", SYSTEM,
        "--egos", "mg",
        "--inputs_dir", inputs_dir,
        "--outputs_dir", outputs_dir,
    ],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("mego failed — see output above")

# ── Locate the output subdirectory (mego appends _1, _2, … automatically) ─────
mg_dirs = sorted(glob.glob(os.path.join(outputs_dir, SYSTEM, "mg_*")))
if not mg_dirs:
    raise RuntimeError(f"mego output not found under {outputs_dir}/{SYSTEM}/mg_*/")
output_dir = mg_dirs[-1]   # most recently created (only one in a fresh Colab)
print(f"\nmego output: {output_dir}")

# ── Copy the force-field files to the working directory ────────────────────────
shutil.copy(os.path.join(output_dir, "topol_mego.top"), "topol_mego.top")
shutil.copy(os.path.join(output_dir, "ffnonbonded.itp"), "ffnonbonded.itp")
print("Files ready: topol_mego.top  ffnonbonded.itp ✓")

## 6 · Simulation parameters

Edit the values in this cell before running it.  The MDP files for energy minimisation
and the production run are generated automatically from the multi-eGO templates.

In [ ]:
# ── Adjust these values ────────────────────────────────────────────────────────
TEMPERATURE_K = 300  # simulation temperature (K)
NSTEPS = 1_000_000  # production steps  (1 M × 5 fs = 5 ns)
# ───────────────────────────────────────────────────────────────────────────────


def fill_mdp(template_path, output_path, subs):
    with open(template_path) as f:
        content = f.read()
    for key, val in subs.items():
        content = content.replace(key, str(val))
    with open(output_path, "w") as f:
        f.write(content)


fill_mdp("multi-ego/mdps/ff_em.mdp", "em.mdp", {})
fill_mdp("multi-ego/mdps/ff_aa.mdp", "ff_aa.mdp", {"_TEMP_": TEMPERATURE_K, "_SET_": NSTEPS})

print(f"em.mdp    → steepest-descent energy minimisation")
print(f"ff_aa.mdp → {NSTEPS:,} steps at {TEMPERATURE_K} K  ({NSTEPS * 0.005 / 1000:.1f} ns)")

## 7 · Energy minimisation  (`gmx mdrun`)

Steepest-descent minimisation with the mg force field.  Removes steric clashes in
the starting structure before dynamics begin.

In [ ]:
!gmx grompp -f em.mdp -c boxed.gro -p topol_mego.top -o em.tpr -maxwarn 1 2>&1 | tail -5
!gmx mdrun -v -deffnm em -ntmpi 1
print("\nMinimisation complete ✓  →  em.gro")

## 8 · NVT production run  (`gmx mdrun`)

Langevin dynamics with the mg force field.  GPU is used automatically when a GPU
runtime is active.  The run produces:

- `run.xtc` — trajectory (needed by `cmdata`)
- `run.tpr` — run-input file (needed by `cmdata`)
- `run.edr` — energy file (used for analysis below)

In [ ]:
!gmx grompp -f ff_aa.mdp -c em.gro -p topol_mego.top -o run.tpr -maxwarn 1 2>&1 | tail -5
!gmx mdrun -v -deffnm run -ntmpi 1
print("\nProduction run complete ✓  →  run.xtc  run.tpr  run.edr")

## 9 · Energy analysis

Extracts potential energy and temperature from the GROMACS energy file and plots them.
Both quantities should be stable by the end of the run.

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt

# Extract potential energy and temperature from the energy file
subprocess.run(
    "printf 'Potential\\nTemperature\\n0\\n' | gmx energy -f run.edr -o energy.xvg", shell=True, capture_output=True
)

# Parse the xvg file (lines starting with # or @ are metadata)
time_ns, potential, temperature = [], [], []
with open("energy.xvg") as f:
    for line in f:
        if line.startswith(("#", "@")):
            continue
        cols = line.split()
        if len(cols) >= 3:
            time_ns.append(float(cols[0]) / 1000)  # ps → ns
            potential.append(float(cols[1]))
            temperature.append(float(cols[2]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(time_ns, potential, color="steelblue", lw=0.8)
ax1.set_xlabel("Time (ns)")
ax1.set_ylabel("Potential energy (kJ/mol)")
ax1.set_title("Potential Energy")
ax1.grid(alpha=0.3)

ax2.plot(time_ns, temperature, color="tomato", lw=0.8)
ax2.axhline(TEMPERATURE_K, color="k", ls="--", lw=1, label=f"{TEMPERATURE_K} K target")
ax2.set_xlabel("Time (ns)")
ax2.set_ylabel("Temperature (K)")
ax2.set_title("Temperature")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("energy_plot.png", dpi=150)
plt.show()

print(f"Mean temperature : {np.mean(temperature):.1f} ± {np.std(temperature):.1f} K")
print(f"Mean pot. energy : {np.mean(potential):.1f} kJ/mol")

## 10 · Download output files

Download the files you need for the next steps of the multi-eGO workflow.

| File | Used for |
|------|----------|
| `run.xtc` | contact histogram extraction with `cmdata` |
| `run.tpr` | required by `cmdata` alongside the trajectory |
| `run.edr` | GROMACS energy file (optional, for further analysis) |
| `energy_plot.png` | visual sanity check |

In [ ]:
from google.colab import files as colab_files
import os

for f in ["run.xtc", "run.tpr", "run.edr", "energy_plot.png"]:
    if os.path.exists(f):
        print(f"Downloading {f}…")
        colab_files.download(f)
    else:
        print(f"  (skipping {f} — not found)")

## Next steps

With `run.xtc` and `run.tpr` in hand, continue the multi-eGO workflow locally:

1. **Extract contact histograms** with `cmdata`:
   ```bash
   cmdata -f run.xtc -s run.tpr
   ```

2. **Convert histograms to contact matrices** with `make_mat`:
   ```bash
   python tools/make_mat/make_mat.py --histo histo/ ...
   ```

3. **Generate the production force field** with `mego`:
   ```bash
   mego --config inputs/SYSTEM/config.yml
   ```

See the [multi-eGO documentation](https://github.com/multi-ego/multi-eGO) for the full workflow.

---

## 11 · Build `cmdata`  *(Option B GROMACS only)*

`cmdata` extracts residue–residue contact histograms from the mg trajectory — the main input
for the `make_mat` step.  It links against the GROMACS library installed in step **Option B**
and requires the legacy API (`-DGMX_INSTALL_LEGACY_API=ON`) that was enabled there.

> **Prerequisite:** you must have run the **Option B** cell above.  Option A does not install
> the GROMACS development headers needed to compile `cmdata`.

The source lives inside the multi-eGO repository (already cloned in step 1).

In [ ]:
import os

# Tell cmake's find_package(GROMACS) where to look
os.environ["GROMACS_DIR"] = "/usr/local/gromacs_mego"

# Configure
!mkdir -p multi-ego/tools/cmdata/build
!cd multi-ego/tools/cmdata/build && cmake ../ 2>&1 | tail -10

# Build and install
!cd multi-ego/tools/cmdata/build && make -j$(nproc) 2>&1 | tail -5
!cd multi-ego/tools/cmdata/build && make install 2>&1 | tail -3

# Verify
!cmdata --help | head -5

### Run `cmdata` on the mg trajectory

With `cmdata` installed you can extract the contact histograms directly in this Colab session.
Adjust `--nskip` (frames to skip between samples) and `--ncontacts` as needed.

In [ ]:
import os

# Source the GROMACS environment so that cmdata can find its shared libraries at runtime
os.environ["PATH"] = "/usr/local/gromacs_mego/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/gromacs_mego/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

NSKIP = 1        # use every Nth frame  (increase to speed up, decrease for finer histograms)

!cmdata -f run.xtc -s run.tpr --nskip {NSKIP}

print("\ncmdata complete — contact histograms written to histo/")

### Run `make_mat.py` — convert histograms to contact matrices

`make_mat.py` reads the `histo/` directory produced by `cmdata` and writes the contact-probability
matrices (`.ndx.h5` files) consumed by the final `mego` step.

| Argument | Value in this notebook |
|----------|----------------------|
| `--target_top` | `topol_mego.top` — topology used for the mg simulation |
| `--mego_top` | `topol.top` — base topology from `gmx pdb2gmx` |
| `--histo` | `histo/` — output directory of `cmdata` |
| `--out` | `mat/` — where the matrices are written |

In [ ]:
import os, multiprocessing, subprocess, sys

CUTOFF      = 0.75   # nm — must match the cutoff used when running cmdata
NUM_THREADS = multiprocessing.cpu_count()

os.makedirs("mat", exist_ok=True)

result = subprocess.run(
    [
        sys.executable,
        "multi-ego/tools/make_mat/make_mat.py",
        "--histo",       "histo/",
        "--target_top",  "topol_mego.top",   # topology used for the mg simulation
        "--mego_top",    "topol.top",         # base topology from gmx pdb2gmx
        "--cutoff",      str(CUTOFF),
        "--out",         "mat/",
        "--num_threads", str(NUM_THREADS),
    ],
    capture_output=True, text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("make_mat.py failed — see output above")

matrices = [f for f in os.listdir("mat") if f.endswith(".h5")]
print(f"\nmatrices written to mat/  ({len(matrices)} files)")
for m in sorted(matrices):
    print(" ", m)